# PhoBERT v7: v6 recipe + hard examples, best of 3 seeds

Setup: **GPU T4**, **Internet On**, secret `HF_TOKEN`.
**Input:** one private dataset with `v7_train.csv` (new: the v6 training rows plus 229 hard examples, comments
that v6 flagged or was unsure about) and `v6_val.csv` (same file as for v6). Columns `text,label`.

Same recipe as v6 (continue v3 for 3 epochs, lr 1e-5, new labels only). Two v6 runs on identical data differed
by 0.04 F1 on the held-out set, so this notebook trains **three seeds** and keeps the one with the best
validation F1-macro. The held-out test half is not on Kaggle and plays no part in the choice.

The chosen model is pushed to the **`v7` branch** (`main` is untouched). Takes about 15 minutes.


In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji
import glob, os, shutil, json
os.makedirs("data/adapt_v6", exist_ok=True)
for split, name in [("train", "v7_train.csv"), ("val", "v6_val.csv")]:
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(matches) == 1, f"Expected exactly one {name}, found: {matches}"
    shutil.copy(matches[0], f"data/adapt_v6/{split}.csv")
shutil.copy("data/adapt_v6/val.csv", "data/adapt_v6/test.csv")  # the real test half never leaves the author's machine
os.makedirs("reports", exist_ok=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
from huggingface_hub import whoami
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
runs = {}
for seed in [42, 1, 2]:
    !python scripts/train_phobert.py --model {repo_id} --data-dir data/adapt_v6 --output-dir models/v7_s{seed} --epochs 3 --lr 1e-5 --batch-size 16 --max-len 128 --seed {seed} --tag _v7_s{seed} --no-push
    runs[seed] = json.load(open(f"reports/phobert_results_v7_s{seed}.json"))["val_f1_macro"]
best = max(runs, key=runs.get)
print("validation F1-macro per seed:", runs, "-> best seed:", best)


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_branch(repo_id, branch="v7", exist_ok=True)
api.upload_folder(folder_path=f"models/v7_s{best}/best", repo_id=repo_id, revision="v7", commit_message=f"v7: v3 adapted to guideline v3.2 with hard examples (seed {best}, best of 3 on validation)")
print(f"https://huggingface.co/{repo_id}/tree/v7")


Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps({"val_f1_macro_per_seed": runs, "best_seed": best, "best": json.load(open(f"reports/phobert_results_v7_s{best}.json"))}, ensure_ascii=False))
